# Clase 084 — Bagging y pasting

Entrenamos el **mismo algoritmo** sobre distintos subconjuntos del training set:
bagging (con reemplazo) vs pasting (sin reemplazo). Evaluamos gratis con
**out-of-bag (OOB)** y cerramos con sampling de features (random subspaces).

Requiere: `numpy`, `scikit-learn`, `matplotlib`.

## 1. Dataset `make_moons`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, load_digits
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score

np.random.seed(42)

X, y = make_moons(n_samples=500, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print('train', X_train.shape, 'test', X_test.shape)

## 2. Árbol único vs bagging de 500 árboles

In [ ]:
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)
acc_tree = accuracy_score(y_test, tree.predict(X_test))

bag = BaggingClassifier(
    DecisionTreeClassifier(random_state=42),
    n_estimators=500, max_samples=100, bootstrap=True,
    random_state=42, n_jobs=1)
bag.fit(X_train, y_train)
acc_bag = accuracy_score(y_test, bag.predict(X_test))

print(f'Árbol único   acc test: {acc_tree:.4f}')
print(f'Bagging (500) acc test: {acc_bag:.4f}')
assert acc_bag >= acc_tree, 'bagging debería igualar o superar al árbol único'
print('assert OK: bagging >= árbol único (reduce varianza)')

## 3. Out-of-bag (OOB): validación gratis

Cada predictor ve ~63% de las instancias; el ~37% restante estima el error de
generalización sin tocar el test set.

In [ ]:
bag_oob = BaggingClassifier(
    DecisionTreeClassifier(random_state=42),
    n_estimators=500, max_samples=100, bootstrap=True,
    oob_score=True, random_state=42, n_jobs=1)
bag_oob.fit(X_train, y_train)
acc_oob_test = accuracy_score(y_test, bag_oob.predict(X_test))

print(f'oob_score_     : {bag_oob.oob_score_:.4f}')
print(f'accuracy test  : {acc_oob_test:.4f}')
assert abs(bag_oob.oob_score_ - acc_oob_test) < 0.05, 'OOB debería aproximar el test'
print('assert OK: OOB aproxima el accuracy de test (diff < 0.05)')

## 4. Bagging vs pasting

In [ ]:
pasting = BaggingClassifier(
    DecisionTreeClassifier(random_state=42),
    n_estimators=500, max_samples=100, bootstrap=False,  # pasting
    random_state=42, n_jobs=1)
pasting.fit(X_train, y_train)
acc_pasting = accuracy_score(y_test, pasting.predict(X_test))
print(f'Bagging acc: {acc_bag:.4f}')
print(f'Pasting acc: {acc_pasting:.4f}')
print('El reemplazo (bagging) añade más diversidad; suele igualar o superar a pasting.')

## 5. Curva de `n_estimators`

In [ ]:
ns = [1, 10, 50, 100, 300, 500]
accs = []
for n in ns:
    b = BaggingClassifier(
        DecisionTreeClassifier(random_state=42),
        n_estimators=n, max_samples=100, bootstrap=True,
        random_state=42, n_jobs=1)
    b.fit(X_train, y_train)
    accs.append(accuracy_score(y_test, b.predict(X_test)))
    print(f'n_estimators={n:4d} -> acc {accs[-1]:.4f}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ns, accs, marker='o', color='#37a')
ax.axhline(acc_tree, ls='--', color='#c33', label=f'árbol único = {acc_tree:.3f}')
ax.set_xlabel('n_estimators')
ax.set_ylabel('accuracy test')
ax.set_title('Bagging: la varianza se promedia con más predictores')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Random subspaces sobre `load_digits` (64 features)

Sampling **solo de features** (todas las instancias): `bootstrap=False`,
`bootstrap_features=True`, `max_features<1.0`.

In [ ]:
Xd, yd = load_digits(return_X_y=True)
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(
    Xd, yd, test_size=0.3, random_state=42, stratify=yd)

subspaces = BaggingClassifier(
    DecisionTreeClassifier(random_state=42),
    n_estimators=100, bootstrap=False, max_samples=1.0,
    bootstrap_features=True, max_features=0.5,
    random_state=42, n_jobs=1)
subspaces.fit(Xd_tr, yd_tr)
print(f'Random subspaces (digits) acc test: '
      f'{accuracy_score(yd_te, subspaces.predict(Xd_te)):.4f}')

## Ejercicios

1. Pedí `oob_score=True` con `bootstrap=False` y observá el `ValueError`: el OOB solo
   existe con reemplazo.
2. Graficá las fronteras de decisión del árbol único y del bagging lado a lado;
   comprobá que la del ensemble es más suave.
3. Random patches: activá `bootstrap=True` **y** `bootstrap_features=True` con
   `max_features=0.5` sobre digits y compará.
4. Sustituí el árbol por `LogisticRegression` (bajo varianza): el bagging casi no
   mueve la aguja. ¿Por qué?

## Conclusiones

- Bagging reduce la **varianza** promediando árboles entrenados sobre bootstraps.
- OOB estima el error de generalización sin separar validación (solo con `bootstrap=True`).
- El reemplazo da más diversidad: bagging suele igualar o superar a pasting.
- Sampling de features (random patches/subspaces) es el puente conceptual hacia
  Random Forests.